In [1]:
import pandas as pd
import numpy as np
import os

cleaned_path = "../data/cleaned/"

customers = pd.read_csv(cleaned_path + "customers.csv")
orders = pd.read_csv(cleaned_path + "orders.csv")
order_items = pd.read_csv(cleaned_path + "order_items.csv")
reviews = pd.read_csv(cleaned_path + "reviews.csv")
products = pd.read_csv(cleaned_path + "products.csv")
category_translation = pd.read_csv(
    cleaned_path + "category_translation.csv"
)

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [2]:
os.makedirs("../data/analysis_outputs", exist_ok=True)

output_path = "../data/analysis_outputs/"

print("Export folder created!")

Export folder created!


In [3]:
# 1. Monthly Revenue

revenue_data = delivered_orders.merge(
    order_items,
    on="order_id"
)

revenue_data["month"] = (
    revenue_data["order_purchase_timestamp"]
    .dt.to_period("M").astype(str)
)

monthly_revenue = (
    revenue_data.groupby("month")["price"]
    .sum()
    .reset_index()
)

monthly_revenue.columns = ["month", "revenue"]


# 2. Category Revenue

product_data = revenue_data.merge(
    products,
    on="product_id",
    how="left"
)

product_data = product_data.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

product_data["category"] = (
    product_data["product_category_name_english"]
    .fillna(product_data["product_category_name"])
    .fillna("Unknown")
)

category_revenue = (
    product_data.groupby("category")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

category_revenue.columns = ["category", "revenue"]


# 3. Customer RFM Segmentation

customer_data = delivered_orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id"
)

customer_data = customer_data.merge(
    order_items[["order_id", "price"]],
    on="order_id"
)

customer_rfm = customer_data.groupby(
    "customer_unique_id"
).agg(
    last_purchase=("order_purchase_timestamp", "max"),
    frequency=("order_id", "nunique"),
    monetary=("price", "sum")
).reset_index()

reference_date = pd.Timestamp("2018-10-01")

customer_rfm["recency"] = (
    reference_date - customer_rfm["last_purchase"]
).dt.days

customer_rfm["r_score"] = pd.qcut(
    customer_rfm["recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

customer_rfm["f_score"] = pd.qcut(
    customer_rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

customer_rfm["m_score"] = pd.qcut(
    customer_rfm["monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)


def segment_customer(row):
    r = row["r_score"]
    f = row["f_score"]

    if r >= 4 and f >= 4:
        return "Champions"
    elif r >= 3 and f >= 4:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Hibernating"
    else:
        return "Potential Loyalists"


customer_rfm["customer_segment"] = customer_rfm.apply(
    segment_customer,
    axis=1
)

segment_counts = (
    customer_rfm["customer_segment"]
    .value_counts()
    .reset_index()
)

segment_counts.columns = [
    "customer_segment",
    "total_customers"
]


# 4. Delivery Performance

delivery_data = delivered_orders.copy()

delivery_data["order_delivered_customer_date"] = pd.to_datetime(
    delivery_data["order_delivered_customer_date"]
)

delivery_data["order_estimated_delivery_date"] = pd.to_datetime(
    delivery_data["order_estimated_delivery_date"]
)

delivery_data = delivery_data.dropna(
    subset=[
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

delivery_data["delivery_status"] = np.where(
    delivery_data["order_delivered_customer_date"]
    <= delivery_data["order_estimated_delivery_date"],
    "On Time",
    "Late"
)

delivery_counts = (
    delivery_data["delivery_status"]
    .value_counts()
    .reset_index()
)

delivery_counts.columns = [
    "delivery_status",
    "total_orders"
]


# 5. Review Distribution

review_distribution = (
    reviews["review_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)

review_distribution.columns = [
    "review_score",
    "total_reviews"
]


# 6. Delivery vs Customer Satisfaction

satisfaction_data = delivery_data.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

delivery_satisfaction = (
    satisfaction_data.groupby("delivery_status")["review_score"]
    .mean()
    .reset_index()
)

delivery_satisfaction.columns = [
    "delivery_status",
    "average_review_score"
]

print("All six analysis datasets prepared!")

All six analysis datasets prepared!


In [4]:
monthly_revenue.to_csv(
    output_path + "monthly_revenue.csv",
    index=False
)

category_revenue.to_csv(
    output_path + "category_revenue.csv",
    index=False
)

segment_counts.to_csv(
    output_path + "customer_segments.csv",
    index=False
)

delivery_counts.to_csv(
    output_path + "delivery_performance.csv",
    index=False
)

review_distribution.to_csv(
    output_path + "review_distribution.csv",
    index=False
)

delivery_satisfaction.to_csv(
    output_path + "delivery_satisfaction.csv",
    index=False
)

print("All six CSV files exported successfully!")

All six CSV files exported successfully!


In [5]:
print(os.listdir(output_path))

['monthly_revenue.csv', 'category_revenue.csv', 'delivery_satisfaction.csv', 'review_distribution.csv', 'delivery_performance.csv', 'customer_segments.csv']


In [7]:
# Geographic Sales Analysis

geo_orders = orders[
    orders["order_status"] == "delivered"
].copy()

# Merge customer information
geo_data = geo_orders.merge(
    customers[
        ["customer_id", "customer_state", "customer_unique_id"]
    ],
    on="customer_id",
    how="left"
)

# Merge order item information
geo_data = geo_data.merge(
    order_items[["order_id", "price"]],
    on="order_id",
    how="left"
)

# Calculate geographic sales
geographic_sales = (
    geo_data.groupby("customer_state")
    .agg(
        total_revenue=("price", "sum"),
        total_orders=("order_id", "nunique"),
        total_customers=("customer_unique_id", "nunique")
    )
    .reset_index()
)

geographic_sales = geographic_sales.rename(
    columns={"customer_state": "state"}
)

geographic_sales = geographic_sales.sort_values(
    "total_revenue",
    ascending=False
)

# Export CSV
geographic_sales.to_csv(
    "../data/analysis_outputs/geographic_sales.csv",
    index=False
)

display(geographic_sales.head(10))

,state,total_revenue,total_orders,total_customers
25,SP,5067633.16,40501,39156
18,RJ,1759651.13,12350,11917
10,MG,1552481.83,11354,11001
22,RS,728897.47,5345,5168
17,PR,666063.51,4923,4769
23,SC,507012.13,3546,3449
4,BA,493584.14,3256,3158
6,DF,296498.41,2080,2019
8,GO,282836.70,1957,1895
7,ES,268643.45,1995,1928


In [9]:
import pandas as pd

payments = pd.read_csv(
    "../data/cleaned/payments.csv"
)

print(payments.shape)
display(payments.head())

(103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [10]:
# ==========================================
# PAYMENT & TRANSACTION ANALYSIS
# ==========================================

payment_analysis = (
    payments.groupby("payment_type")
    .agg(
        total_payment_value=("payment_value", "sum"),
        total_transactions=("payment_value", "count"),
        total_orders=("order_id", "nunique"),
        average_payment=("payment_value", "mean"),
        average_installments=("payment_installments", "mean")
    )
    .reset_index()
)

payment_analysis = payment_analysis.sort_values(
    "total_payment_value",
    ascending=False
)

payment_analysis.to_csv(
    "../data/analysis_outputs/payment_analysis.csv",
    index=False
)

display(payment_analysis)

,payment_type,total_payment_value,total_transactions,total_orders,average_payment,average_installments
1,credit_card,12542084.19,76795,76505,163.319021,3.507155
0,boleto,2869361.27,19784,19784,145.034435,1.000000
4,voucher,379436.87,5775,3866,65.703354,1.000000
2,debit_card,217989.79,1529,1528,142.570170,1.000000
3,not_defined,0.00,3,3,0.000000,1.000000


In [11]:
import pandas as pd

# Load cleaned datasets
order_items = pd.read_csv(
    "../data/cleaned/order_items.csv"
)

sellers = pd.read_csv(
    "../data/cleaned/sellers.csv"
)

# Merge order items with seller information
seller_data = order_items.merge(
    sellers[
        ["seller_id", "seller_city", "seller_state"]
    ],
    on="seller_id",
    how="left"
)

# Calculate seller performance
seller_analysis = (
    seller_data.groupby(
        ["seller_id", "seller_city", "seller_state"]
    )
    .agg(
        total_revenue=("price", "sum"),
        total_orders=("order_id", "nunique"),
        total_items_sold=("order_id", "count"),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

# Calculate average revenue per seller-associated order
seller_analysis["average_order_value"] = (
    seller_analysis["total_revenue"]
    / seller_analysis["total_orders"]
)

# Sort sellers by revenue
seller_analysis = seller_analysis.sort_values(
    "total_revenue",
    ascending=False
)

# Export CSV
seller_analysis.to_csv(
    "../data/analysis_outputs/seller_analysis.csv",
    index=False
)

display(seller_analysis.head(10))

,seller_id,seller_city,seller_state,total_revenue,total_orders,total_items_sold,average_item_price,average_order_value
857,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,229472.63,1132,1156,198.505735,202.714337
1013,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,222776.05,358,410,543.356220,622.279469
881,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,200472.92,1806,1987,100.892260,111.003832
3024,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,194042.03,585,586,331.129744,331.695778
1535,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,187923.89,982,1364,137.774113,191.368523
1560,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,176431.87,336,340,518.917265,525.094851
2643,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,160236.57,1314,1551,103.311779,121.945639
1505,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,141745.53,1160,1171,121.046567,122.194422
192,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,138968.55,915,1428,97.316912,151.878197
1824,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,135171.70,1287,1499,90.174583,105.028516
